# Fundamentals 01 - Tool API

**Historia:** antes de hablar de agentes, entendemos la unidad minima de Agentic Systems: una `Tool`.

En `2.4.3` ademas materializamos el contrato alrededor de esa tool: input/output Pydantic, expectativa de ejecucion y policy reutilizable para quien la convierta en agente.

In [ ]:
import agentic_systems as toolkit
from pydantic import BaseModel

PRETTY = False  # Cambia a True para usar Rich; False imprime texto plano estable y reproducible.

## Escenario didactico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para que puedas comparar la API sin cambiar de caso cada vez:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```



## Parametros de `RunPolicy`

`RunPolicy` declara c?mo debe comportarse una ejecucion antes de llamar al agente o al runtime. No es metadata decorativa: limita loops, define reparacion, controla trazas y hace que el resultado sea evaluable.

| Parametro | Que controla | Uso recomendado |
|---|---|---|
| `max_turns` | N?mero m?ximo de turnos internos del agente. | Mantenerlo bajo en notebooks para evitar loops largos. |
| `max_tool_calls` | N?mero m?ximo de llamadas a tools. | Declararlo cuando el ejercicio espera tools concretas. |
| `max_tokens` | L?mite de tokens del modelo cuando el provider lo soporta. | ?til en providers LM; puede quedar `None` en `python-direct`. |
| `temperature` | Aleatoriedad del modelo. | `0.0` para tutoriales reproducibles; `None` delega al provider. |
| `tool_choice` | Estrategia de seleccion de tools, por ejemplo `auto`. | `auto` cuando el agente decide; explicito cuando quieres forzar una tool. |
| `repair` | Permite reparacion autom?tica de salidas o tool calls invalidas. | `True` para UX robusta; `False` si quieres ver fallos crudos. |
| `max_repairs` | M?ximo de intentos de reparacion. | `1` o `2` en tutoriales para mostrar control sin ocultar errores. |
| `finalize` | Que hacer al agotar turnos, por ejemplo `on_max_turns`. | Mantenerlo explicito en agentes LM evaluables. |
| `trace` | Nivel de trazabilidad (`compact`, `debug`, etc.). | `compact` para notebooks; `debug` solo para diagnostico. |
| `strict` | Si el contrato debe aplicarse de forma estricta. | `True` para ensenar API y evitar ambiguedad. |


In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

# La estructura solicitada por el usuario se materializa como datos simples.
# No es un parser ni una respuesta precocinada: solo representa la seccion `Dime:`.
REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

toolkit.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario didactico  visible")


## 1) Crear una tool rapida con `@toolkit.tool`

Aqui no hay agente todavia. Solo hay una funcion Python convertida en una `Tool` ejecutable.

In [ ]:
@toolkit.tool
def sumar(a: int, b: int) -> dict:
    """Suma dos numeros y regresa output estructurado."""
    return {
        "operation": "sumar",
        "result": a + b,
        "explanation": f"{a} + {b} = {a + b}",
    }

quick_result = sumar.run({"a": 17, "b": 25})
toolkit.human_result(
    quick_result,
    title="Human result  tool rapida",
    expected_tools=toolkit.expect.exactly("sumar"),
    pretty=PRETTY,
)

## 2) Crear una tool explicita con schema

Usa `toolkit.Tool(...)` cuando quieras dejar el contrato de entrada/salida visible desde el objeto.

In [ ]:
class TwoNumbers(BaseModel):
    a: int
    b: int


class OperationOutput(BaseModel):
    operation: str
    result: int
    explanation: str


def restar_fn(payload: TwoNumbers) -> OperationOutput:
    value = payload.a - payload.b
    return OperationOutput(operation="restar", result=value, explanation=f"{payload.a} - {payload.b} = {value}")


restar = toolkit.Tool(
    restar_fn,
    name="restar",
    description="Resta dos numeros y devuelve salida estructurada.",
    input=TwoNumbers,
    output=OperationOutput,
)

contract_result = restar.run({"a": 50, "b": 8})
toolkit.human_result(contract_result, title="Human result  tool con Pydantic", expected_tools=toolkit.expect.exactly("restar"), pretty=PRETTY)

## 3) Materializar contrato + policy

El contrato responde **que debe pasar**. La policy responde **con que limites debe ejecutarse**.

Este spec no ejecuta nada por si solo; es metadata validable y reusable para agentes, sistemas, skills o evals.

In [ ]:
sumar_spec = toolkit.ContractPolicySpec(
    name="fundamentals.sumar_once",
    description="Debe llamar sumar exactamente una vez y conservar salida exitosa.",
    contract=toolkit.AgentContract(
        must_call=["sumar"],
        tool_expectation=toolkit.expect.exactly("sumar"),
        completion="when_required_tools_satisfied",
        failure_policy="no_unresolved",
        expected_tool_outputs={"sumar": {"operation": "sumar", "result": 42}},
    ),
    policy=toolkit.RunPolicy(
        max_turns=4,
        max_tool_calls=1,
        temperature=0.0,
        tool_choice="sumar",
        finalize="after_required_tools",
    ),
    tags=["fundamentals", "tool", "contract"],
)

available_tools = [sumar.name, restar.name]
toolkit.show({
    "spec": sumar_spec.describe(),
    "static_check": sumar_spec.check(available_tools=available_tools).to_dict(),
})

## 4) Validacion declarativa antes de ejecutar

Aqui provocamos un contrato imposible para ver el error sin llamar ningun modelo ni tool.

In [ ]:
impossible = toolkit.ContractPolicySpec(
    name="fundamentals.invalid_budget",
    contract=toolkit.AgentContract(must_call=["sumar", "restar"]),
    policy=toolkit.RunPolicy(max_tool_calls=1),
)

toolkit.show(impossible.check(available_tools=available_tools).to_dict())

## 5) Resolver el escenario didactico solo con tools

Aqui no hay parser ni agente. Leemos el prompt y ejecutamos las operaciones con tools, una por una.

La parte didactica es ver el patron:

```text
Tool.run(input)  salida estructurada  siguiente Tool.run(...)
```


In [ ]:
@toolkit.tool
def multiplicar(a: int, b: int) -> dict:
    """Multiplica dos numeros."""
    return {"operation": "multiplicar", "result": a * b, "explanation": f"{a}  {b} = {a * b}"}


@toolkit.tool
def dividir(a: int, b: int) -> dict:
    """Divide dos numeros."""
    if b == 0:
        raise ValueError("No se puede dividir entre cero.")
    value = a / b
    result = int(value) if value.is_integer() else value
    return {"operation": "dividir", "result": result, "explanation": f"{a}  {b} = {result}"}


# Directo del prompt del usuario:
# "Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2."
step_1 = sumar.run({"a": 10, "b": 20})
step_2 = restar.run({"a": step_1.data["result"], "b": 9})
step_3 = multiplicar.run({"a": step_2.data["result"], "b": 4})
step_4 = dividir.run({"a": step_3.data["result"], "b": 2})

procedure = [step.data["explanation"] for step in [step_1, step_2, step_3, step_4]]
answer = {
    "procedimiento": procedure,
    "resultado_final": step_4.data["result"],
}

toolkit.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
    "respuesta_materializada": answer,
})


## Lo importante

- `Tool` valida input/output.
- `AgentContract` declara expectativas de comportamiento.
- `RunPolicy` declara limites de ejecucion.
- `ContractPolicySpec` empaqueta ambas cosas para reuso.

## Coverage API de este notebook

Esta tabla deja explicito que parte de Agentic Systems queda materializada aqui.

In [ ]:
api_coverage = [
    {
        "api": "@toolkit.tool",
        "description": "Construye herramientas declarativas para reutilizarlas en agentes y grafos."
    },
    {
        "api": "toolkit.Tool",
        "description": "Materializa la tool como primitiva reusable y auditable."
    },
    {
        "api": "Pydantic input/output",
        "description": "Tipa la entrada y salida para que el contrato sea verificable."
    },
    {
        "api": "ContractPolicySpec",
        "description": "Une contrato y policy para validar intencion antes de ejecutar."
    },
    {
        "api": "Tool.run",
        "description": "Ejecuta la tool de forma directa para ver su boundary aislado."
    },
    {
        "api": "human_result",
        "description": "Renderiza la salida humana sin perder la evidencia interna."
    },
    {
        "api": "manual tool chaining",
        "description": "Compone tools paso a paso para ensenar el flujo sin agente."
    }
]

toolkit.show({'notebook': '01_tool_api.ipynb', 'api_coverage': api_coverage})


## Simbolos API explicados

Este notebook se alinea con `docs/API.md` y ensena estos simbolos publicos:

- `Tool / toolkit.tool`: Primitiva ejecutable y decorator publico.
- `PublicToolRegistry`: Registro publico de tools cuando se necesita composici?n.
- `toolkit.expect`: Declaraci?n de expectativas de tools.
- `ToolExpectationValue / normalize_tool_expectation`: Normalizaci?n publica de expectativas.
- `validate_tool_expectation`: Validador publico de ejecucion esperada.
- `ValidationIssue / ValidationResult`: Tipos publicos de validaci?n.
- `AgentContract / ContractPolicySpec / RunPolicy`: Contrato y policy aplicables a tools.

